Tesla Deliveries ML Pipeline   
End-to-end: EDA -> preprocessing -> feature engineering ->
regression modeling -> hyperparameter tuning -> forecasting

In [1]:
import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.inspection import permutation_importance
import joblib

In [2]:
# Optional models
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

try:
    import statsmodels.api as sm
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    STATS_MODELS_AVAILABLE = True
except Exception:
    STATS_MODELS_AVAILABLE = False

In [3]:
# ---------------------------
# 1. CONFIG
# ---------------------------
FILE_PATH = "tesla_deliveries_dataset_2015_2025.csv"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

In [4]:
# ---------------------------
# 2. LOAD DATA
# ---------------------------
df = pd.read_csv("tesla_deliveries_dataset_2015_2025.csv")

print("\nDataset shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nPreview:\n", df.head())


Dataset shape: (2640, 12)

Columns:
 ['Year', 'Month', 'Region', 'Model', 'Estimated_Deliveries', 'Production_Units', 'Avg_Price_USD', 'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons', 'Source_Type', 'Charging_Stations']

Preview:
    Year  Month         Region    Model  Estimated_Deliveries  \
0  2023      5         Europe  Model S                 17646   
1  2015      2           Asia  Model X                  3797   
2  2019      1  North America  Model X                  8411   
3  2021      2  North America  Model 3                  6555   
4  2016     12    Middle East  Model Y                 12374   

   Production_Units  Avg_Price_USD  Battery_Capacity_kWh  Range_km  \
0             17922       92874.27                   120       704   
1              4164       62205.65                    75       438   
2              9189      117887.32                    82       480   
3              7311       89294.91                   120       712   
4             13537      114

In [5]:
# ---------------------------
# 3. BASIC CLEANING
# ---------------------------
df.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in df.columns]

# remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# strip text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

In [6]:

# ---------------------------
# 4. AUTO-DETECT DATE COLUMN
# ---------------------------
date_candidates = [c for c in df.columns if any(k in c for k in ["date", "month", "quarter", "year", "time"])]
date_col = None

for c in date_candidates:
    try:
        parsed = pd.to_datetime(df[c], errors="coerce")
        if parsed.notna().sum() > max(0.6 * len(df), 3):
            date_col = c
            df[c] = parsed
            break
    except Exception:
        pass

if date_col is None:
    # try the first object-like column as date if it looks like dates
    for c in df.columns:
        if df[c].dtype == "object":
            parsed = pd.to_datetime(df[c], errors="coerce")
            if parsed.notna().sum() > max(0.6 * len(df), 3):
                date_col = c
                df[c] = parsed
                break

print("\nDetected date column:", date_col)


Detected date column: year


In [7]:

# ---------------------------
# 5. AUTO-DETECT TARGET COLUMN
# ---------------------------
target_candidates = [c for c in df.columns if any(k in c for k in [
    "delivery", "deliveries", "sales", "sold", "target", "revenue", "price", "cost", "volume"
])]

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

target_col = None
for c in target_candidates:
    if c in numeric_cols:
        target_col = c
        break

if target_col is None and len(numeric_cols) > 0:
    target_col = numeric_cols[0]

print("Detected target column:", target_col)


Detected target column: estimated_deliveries


In [9]:
# ---------------------------
# 6. EDA
# ---------------------------
print("\nMissing values:\n", df.isna().sum().sort_values(ascending=False).head(20))
print("\nNumeric summary:\n", df.describe().T)

# Save summary tables
df.describe(include="all").to_csv(os.path.join(OUTPUT_DIR, "eda_summary.csv"))

plt.figure(figsize=(12, 6))
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
if len(missing) > 0:
    sns.barplot(x=missing.values, y=missing.index, palette="viridis")
    plt.title("Missing Values by Column")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "missing_values.png"), dpi=200)
    plt.close()

if target_col is not None and target_col in df.columns:
    plt.figure(figsize=(10, 5))
    sns.histplot(df[target_col].dropna(), kde=True, bins=30, color="steelblue")
    plt.title(f"Distribution of {target_col}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "target_distribution.png"), dpi=200)
    plt.close()

# Correlation heatmap
num_df = df.select_dtypes(include=np.number)
if num_df.shape[1] > 1:
    plt.figure(figsize=(12, 8))
    sns.heatmap(num_df.corr(), cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "correlation_heatmap.png"), dpi=200)
    plt.close()


Missing values:
 year                    0
month                   0
region                  0
model                   0
estimated_deliveries    0
production_units        0
avg_price_usd           0
battery_capacity_kwh    0
range_km                0
co2_saved_tons          0
source_type             0
charging_stations       0
dtype: int64

Numeric summary:
                        count                           mean  \
year                    2640  1970-01-01 00:00:00.000002020   
month                 2640.0                            6.5   
estimated_deliveries  2640.0                    9922.199621   
production_units      2640.0                   10655.847348   
avg_price_usd         2640.0                    84907.34033   
battery_capacity_kwh  2640.0                       87.05947   
range_km              2640.0                     500.257576   
co2_saved_tons        2640.0                     744.076989   
charging_stations     2640.0                    8932.133712   

       

<Figure size 1200x600 with 0 Axes>

In [15]:
# ---------------------------
# 7. FEATURE ENGINEERING
# ---------------------------
data = df.copy()

# date-derived features
if date_col is not None and date_col in data.columns:
    dt = pd.to_datetime(data[date_col], errors="coerce")

    if dt.notna().sum() > 0:
        data["year"] = dt.dt.year
        data["month"] = dt.dt.month
        data["day"] = dt.dt.day
        data["dayofweek"] = dt.dt.dayofweek
        data["quarter"] = dt.dt.quarter
        data["is_month_start"] = dt.dt.is_month_start.astype(int)
        data["is_month_end"] = dt.dt.is_month_end.astype(int)

        data[date_col] = dt
    else:
        print(f"Warning: {date_col} has no valid dates. Skipping date features.")

# sort by date for temporal features
if date_col is not None and date_col in data.columns:
    if pd.to_datetime(data[date_col], errors="coerce").notna().sum() > 0:
        data = data.sort_values(date_col).reset_index(drop=True)

# lag/rolling features for target
if target_col is not None and target_col in data.columns:
    for lag in [1, 2, 3, 4]:
        data[f"{target_col}_lag_{lag}"] = data[target_col].shift(lag)

    for win in [3, 6]:
        data[f"{target_col}_roll_mean_{win}"] = data[target_col].shift(1).rolling(win).mean()
        data[f"{target_col}_roll_std_{win}"] = data[target_col].shift(1).rolling(win).std()

# simple interaction features for numeric variables
num_cols = data.select_dtypes(include=np.number).columns.tolist()
for i in range(min(3, len(num_cols))):
    for j in range(i + 1, min(3, len(num_cols))):
        c1, c2 = num_cols[i], num_cols[j]
        data[f"{c1}_x_{c2}"] = data[c1] * data[c2]

In [11]:
# ---------------------------
# 8. DEFINE REGRESSION DATASET
# ---------------------------
# Drop rows with missing target
data = data.dropna(subset=[target_col]).reset_index(drop=True)

# For forecasting/regression, keep only rows where lag features are available
data_model = data.dropna().reset_index(drop=True)

# Separate features and target
y = data_model[target_col].copy()
X = data_model.drop(columns=[target_col])

# Remove raw date column from features after extracting calendar features
if date_col is not None and date_col in X.columns:
    X = X.drop(columns=[date_col])

# Identify categorical/numeric columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("\nFeature columns:", len(X.columns))
print("Numeric columns:", len(num_cols))
print("Categorical columns:", len(cat_cols))


Feature columns: 10
Numeric columns: 7
Categorical columns: 3


In [16]:
# ---------------------------
# 9. TRAIN-TEST SPLIT
# ---------------------------
if date_col is not None:
    # Time-based split to prevent leakage
    split_idx = int(len(X) * (1 - TEST_SIZE))
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
